# Notebook 05 — ADCD-Net evaluation (Week 5)

**Goal:** score the probability maps from notebook 04 against my answer-key masks, and say how good or bad ADCD-Net is on my forged bank statements, with confidence intervals.

**Kaggle settings:** Accelerator = **None** (no GPU needed, saves quota). Internet can be off.
**Inputs to attach (Add Input → Datasets):** `PATH_TO_TEXTCTRL_RUN_DATASET` and `PATH_TO_ADCD_RUN_OUTPUTS_DATASET` (the zip from notebook 04, uploaded as a dataset).

**This file so far covers P6 (headline metrics) only.** P7 (breakdowns, failure gallery) and P8a (size trend) are added later, below.

| Section | ⏱ Time | What you get |
|---|---|---|
| 1. CONFIG | < 5 s | settings, output folder |
| 2. Find the inputs | < 10 s | paths and file counts |
| 3. Per-image pass over the maps | 1–2 min | counts for every page and threshold |
| 4. Precision / recall / F1 / IoU + detection rate | < 30 s | table with 95% CIs |
| 5. Pixel AUC, ROC, best-threshold F1 | 2–4 min (bootstrap) | ROC curve figure |
| 6. Image-level AUC and false alarms | < 30 s | table with 95% CIs |
| 7. Summary table + bar chart, save CSVs | < 30 s | `summary.csv`, `per_image_metrics.csv`, PNG figures |

**Total: about 5–8 minutes.** Because it needs no GPU, you can use **Save & Run All** on this notebook at the end to keep its outputs.

## 1. CONFIG (P6)

**WHY:** same rule as notebook 04: every setting in one place, seeds fixed, settings saved as `config.json` next to the results. This notebook writes into its **own new folder** (`adcd_eval_<date>_<time>`), because the inputs are read-only and old runs are never overwritten.

⏱ < 5 s. ✅ The output folder path and the settings printed.

In [ ]:
# Standard tools: files, json, randomness, timing, dates
import os, json, random, glob, time
from datetime import datetime
# numpy = maths on arrays, pandas = tables
import numpy as np
import pandas as pd
# cv2 reads and resizes the mask images (same tool the ADCD-Net loader uses)
import cv2
# rankdata is used for a fast AUC; roc_curve and roc_auc_score for curves and image-level AUC
from scipy.stats import rankdata
from sklearn.metrics import roc_curve, roc_auc_score
# matplotlib draws the figures (shown inside the notebook, and saved as PNG)
import matplotlib.pyplot as plt

# All settings in one dictionary
CONFIG = {
    # Seed for random, numpy (same seed = same random choices every run)
    "seed": 42,
    # Probability cut-off: a pixel with probability >= 0.5 counts as "tampered"
    "threshold": 0.5,
    # ADCD-Net shrinks the longest side of each page to this many pixels
    "val_max_size": 512,
    # Number of bootstrap resamples for the 95% confidence intervals
    "N_BOOT": 1000,
    # Sizes and orderings for the size-trend analysis (used later, in P8a)
    "SIZES": list(range(10, 101, 10)),
    "N_ORDERINGS": 20,
    # Tampered input variants we score (originals are used for false alarms only)
    "eval_variants": ["png", "jpg90"],
    # The two answer keys: whole pasted rectangle (primary) and pixels that really changed (strict)
    "gt_kinds": ["box", "diff"],
    # Share of non-tampered pixels kept for the pixel AUC (the rest are skipped to save memory)
    "neg_frac": 0.01,
    # Thresholds tried for the "best-threshold F1" (0.05, 0.10, ..., 0.95)
    "threshold_grid": [round(0.05 * k, 2) for k in range(1, 20)],
    # Tiny number added to denominators (the same value main.py uses)
    "eps": 1e-8,
}
# Seed Python's random module and numpy
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

# Kaggle's writable folder
WORK = "/kaggle/working"
# New output folder for this notebook, named with today's date and time
EVAL_NAME = "adcd_eval_" + datetime.now().strftime("%Y%m%d_%H%M")
EVAL_DIR = f"{WORK}/adcd_runs/{EVAL_NAME}"
# Create it, with a figures sub-folder
os.makedirs(f"{EVAL_DIR}/figures", exist_ok=True)
# Save the settings as config.json
with open(f"{EVAL_DIR}/config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

# The thresholds as a numpy array, and the position of 0.5 in it
THRESHOLDS = np.array(CONFIG["threshold_grid"])
I50 = int(np.argmin(np.abs(THRESHOLDS - CONFIG["threshold"])))

print("Output folder:", EVAL_DIR)
print("Settings     :", {k: CONFIG[k] for k in ["seed", "threshold", "N_BOOT", "neg_frac"]})
print("Thresholds   :", THRESHOLDS.tolist(), "| official threshold is index", I50, "=", THRESHOLDS[I50])

## 2. Find the inputs (P6)

**WHY:** we need three things: the answer-key masks and `results.csv` (from `PATH_TO_TEXTCTRL_RUN_DATASET`), and the probability maps plus `scores.csv` (from the notebook-04 zip, uploaded as the dataset `PATH_TO_ADCD_RUN_OUTPUTS_DATASET`). Kaggle may put the zip's inner folder one level down, so we search for `scores.csv` instead of guessing, and print what we found. Maps are only **counted**, never opened by hand.

⏱ < 10 s. ✅ Both dataset paths, `100` maps in each of `png`, `jpg90`, `orig`, `100` box masks and `100` diff masks, and a `scores.csv` with 300 rows (3 variants x 100).

In [ ]:
# Function: find a dataset folder by trying both Kaggle path styles
def find_dataset(slug):
    for candidate in [f"/kaggle/input/datasets/PATH_TO_KAGGLE_USERNAME/{slug}", f"/kaggle/input/{slug}"]:
        if os.path.isdir(candidate):
            return candidate
    # Not found: show what is mounted (max 20 names)
    print("⚠️ Not found:", slug, "| mounted:", sorted(glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/datasets/*/*"))[:20])
    return None

# Function: first file with this name at most 3 folder levels below `root`
def find_file(root, name):
    for pattern in [name, f"*/{name}", f"*/*/{name}", f"*/*/*/{name}"]:
        hits = sorted(glob.glob(os.path.join(root, pattern)))
        if hits:
            return hits[0]
    return None

# Locate the two datasets
TC_DS = find_dataset("PATH_TO_TEXTCTRL_RUN_DATASET")
OUT_DS = find_dataset("PATH_TO_ADCD_RUN_OUTPUTS_DATASET")
print("PATH_TO_TEXTCTRL_RUN_DATASET :", TC_DS)
print("PATH_TO_ADCD_RUN_OUTPUTS_DATASET:", OUT_DS)

# The Week 4 run folder is wherever results.csv lives
SRC_RUN = os.path.dirname(find_file(TC_DS, "results.csv"))
# The notebook-04 run folder is wherever scores.csv lives
RUN_IN = os.path.dirname(find_file(OUT_DS, "scores.csv"))
PROBS_DIR = f"{RUN_IN}/probs"
if SRC_RUN != TC_DS:
    print("⚠️ textctrl run is nested, using:", SRC_RUN)
if RUN_IN != OUT_DS:
    print("⚠️ adcd run is nested, using:", RUN_IN)

# Count the maps (only counted, not opened) and the masks
for variant in ["png", "jpg90", "orig"]:
    n = len([f for f in os.listdir(f"{PROBS_DIR}/{variant}") if f.endswith(".npy")])
    print(f"maps in probs/{variant:5s}: {n}")
for kind in CONFIG["gt_kinds"]:
    print(f"masks in masks/{kind:4s}: {len(os.listdir(f'{SRC_RUN}/masks/{kind}'))}")

# Load the two tables
scores = pd.read_csv(f"{RUN_IN}/scores.csv")
results = pd.read_csv(f"{SRC_RUN}/results.csv")
print("\nscores.csv :", scores.shape, "| variants:", scores["variant"].value_counts().to_dict())
print("results.csv:", results.shape)
print("scores columns:", list(scores.columns))

## 3. One pass over the maps: count pixels for every page and threshold (P6)

**WHY:** every metric in this notebook is built from a few counts per page. Doing one careful pass and keeping the counts means the later sections only do arithmetic. For each tampered variant (`png`, `jpg90`) and each answer key (`box`, `diff`) the pass:
1. loads the page's probability map (it is at the model's size, at most 512 px on the long side);
2. shrinks the answer key to the same size with *nearest-neighbour* resizing (this is what the repo's loader does, so the pixels line up exactly);
3. sorts the probabilities of the tampered pixels and of the clean pixels, so that for **any** threshold we can read off how many pixels are above it (TP = tampered pixels above the cut, FP = clean pixels above it);
4. keeps all tampered pixels and a random 1% of clean pixels for the pixel AUC.

**Heads-up about the strict `diff` key:** shrinking a thin mask about 4.8x with nearest-neighbour can lose many of its pixels (or all of them). Pages whose shrunken `diff` key is empty have an undefined F1; they are left out of the `diff` numbers and counted.

**Consistency check:** the notebook-04 script already computed an F1 for the `box` key at 0.5. We recompute it from the saved float16 maps; the two should match up to tiny rounding (float16 near the 0.5 cut).

⏱ 1–2 min. ✅ `Done`, the mask-size line, a consistency line like `max abs difference ... < 0.01`, and the number of pages with an empty `diff` key.

In [ ]:
# Random generator for the 1% sample of clean pixels (seeded, so the sample is the same every run)
rng_sample = np.random.default_rng(CONFIG["seed"])
# Cache of answer keys already shrunk to the model's size (both variants share the same pages)
GT_CACHE = {}

# Function: load an answer key, make it True/False, shrink it to (h, w) like the repo does
def get_gt(kind, image, h, w):
    key = (kind, image)
    if key not in GT_CACHE:
        # Read the mask as grey; any non-zero pixel is "tampered"
        m = (cv2.imread(f"{SRC_RUN}/masks/{kind}/{image}.png", 0) != 0).astype(np.uint8)
        # Nearest-neighbour shrink to the model's size (cv2 wants width first)
        m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
        GT_CACHE[key] = m.astype(bool)
    return GT_CACHE[key]

# STORE[(variant, key)] holds the counts for every page, in the order of `images`
STORE = {}
# Start a stopwatch
t0 = time.time()
for variant in CONFIG["eval_variants"]:
    # The rows of scores.csv for this variant, sorted by page name
    rows_v = scores[scores["variant"] == variant].sort_values("image").reset_index(drop=True)
    for kind in CONFIG["gt_kinds"]:
        STORE[(variant, kind)] = {"images": rows_v["image"].tolist(), "tp": [], "fp": [], "gt_pixels": [],
                                  "page_pixels": [], "pos": [], "neg": []}
    for rec in rows_v.itertuples():
        # Load the probability map (float16 on disk -> float32 in memory)
        prob = np.load(f"{PROBS_DIR}/{variant}/{rec.image}.npy").astype(np.float32)
        # The map must have the size recorded by the inference script
        assert prob.shape == (rec.h, rec.w), f"{rec.image}: map {prob.shape} vs ({rec.h}, {rec.w})"
        for kind in CONFIG["gt_kinds"]:
            # The answer key at the map's size
            gt = get_gt(kind, rec.image, rec.h, rec.w)
            # Probabilities of the tampered pixels and of the clean pixels
            pos_vals = prob[gt]
            neg_vals = prob[~gt]
            # Sort both, so we can count how many are >= each threshold
            pos_sorted = np.sort(pos_vals)
            neg_sorted = np.sort(neg_vals)
            # TP(t) = tampered pixels with probability >= t ; FP(t) = clean pixels with probability >= t
            tp = len(pos_sorted) - np.searchsorted(pos_sorted, THRESHOLDS, side="left")
            fp = len(neg_sorted) - np.searchsorted(neg_sorted, THRESHOLDS, side="left")
            # A random 1% of the clean pixels, kept for the pixel AUC
            k = int(round(CONFIG["neg_frac"] * len(neg_vals)))
            neg_sub = rng_sample.choice(neg_vals, size=k, replace=False) if k > 0 else np.array([], dtype=np.float32)
            # Save everything for this page
            s = STORE[(variant, kind)]
            s["tp"].append(tp)
            s["fp"].append(fp)
            s["gt_pixels"].append(int(gt.sum()))
            s["page_pixels"].append(int(gt.size))
            s["pos"].append(pos_vals)
            s["neg"].append(neg_sub)
# Turn the lists of counts into arrays: shape (pages, thresholds)
for s in STORE.values():
    s["tp"] = np.array(s["tp"])
    s["fp"] = np.array(s["fp"])
    s["gt_pixels"] = np.array(s["gt_pixels"])
    s["page_pixels"] = np.array(s["page_pixels"])
print(f"Done in {time.time() - t0:.0f} s.")

# Fact: how much of the model's page the answer key covers (why pixel accuracy is useless)
box_share = STORE[("png", "box")]["gt_pixels"] / STORE[("png", "box")]["page_pixels"]
print(f"Box covers a median {np.median(box_share) * 100:.2f}% of the page (range {box_share.min() * 100:.2f}% to {box_share.max() * 100:.2f}%).")

# How many pages have an empty answer key after shrinking
for kind in CONFIG["gt_kinds"]:
    n_empty = int((STORE[("png", kind)]["gt_pixels"] == 0).sum())
    print(f"Pages with an empty '{kind}' key after shrinking: {n_empty} of {len(STORE[('png', kind)]['images'])}")

# Consistency check: recompute the box F1 at 0.5 and compare with the number notebook 04 stored
eps = CONFIG["eps"]
for variant in CONFIG["eval_variants"]:
    s = STORE[(variant, "box")]
    tp, pred, gt_n = s["tp"][:, I50], s["tp"][:, I50] + s["fp"][:, I50], s["gt_pixels"]
    p, r = tp / (pred + eps), tp / (gt_n + eps)
    f1_new = 2 * p * r / (p + r + eps)
    f1_old = scores[scores["variant"] == variant].sort_values("image")["f1"].to_numpy()
    diff = np.abs(f1_new - f1_old)
    print(f"{variant}: recomputed box F1 vs notebook 04: max abs difference {diff.max():.4f}, mean {diff.mean():.4f}"
          + ("   ⚠️ larger than expected" if diff.max() > 0.02 else "   OK"))

## 4. Precision, recall, F1, IoU and detection rate at 0.5 (P6)

**The idea, step by step.** For one page we compare two black-and-white images: the **answer key** (white = tampered) and the **prediction** (white = probability ≥ 0.5). Count pixels:
- **TP** = white in both (found the edit), **FP** = white only in the prediction (false alarm), **FN** = white only in the key (missed).

**WHAT → WHY → HOW** of each number:
- **Precision** — of the pixels the model flagged, how many were really tampered? $P = \frac{TP}{TP+FP}$
- **Recall** — of the really tampered pixels, how many did the model find? $R = \frac{TP}{TP+FN}$
- **F1** — one number balancing both (harmonic mean: it is low if either is low). $F1 = \frac{2PR}{P+R}$
- **IoU** — overlap divided by union, stricter than F1. $IoU = \frac{TP}{TP+FP+FN}$

*Tiny example:* the box has 100 pixels, the model flags 50, and 20 of those are inside the box. TP = 20, FP = 30, FN = 80, so P = 0.40, R = 0.20, F1 = 0.27, IoU = 0.15.

**Official-style F1** (what the paper's evaluation does): compute P, R, F1 **for each page separately**, then **average the page F1s**. Every page counts equally. A page where nothing is flagged scores F1 = 0.

**Detection rate** — the share of tampered pages where the prediction touches the key at all (TP > 0). It is a very lenient test.

**Pixel accuracy is useless here:** the box covers well under 1% of the page (section 3 printed the exact share), so a model that flags nothing is "99%+ correct".

**95% confidence interval (bootstrap):** resample the pages with replacement 1000 times, recompute the average each time, keep the 2.5th and 97.5th percentiles. With only 100 pages, a bare average hides how much it could wobble.

Two answer keys: **box** (the whole pasted rectangle, primary) and **diff** (only pixels whose grey value really changed, stricter). Two inputs: **png** and **jpg90** (the JPEG-q90 copy).

⏱ < 30 s. ✅ A table with 4 rows (png/box, png/diff, jpg90/box, jpg90/diff). Expect **low F1** (we predicted this) and a detection rate that is much higher than F1.

In [ ]:
# Function: 95% bootstrap confidence interval for the mean of a list of numbers (NaN = not defined, skipped)
def boot_mean_ci(values, n_boot=CONFIG["N_BOOT"], seed=CONFIG["seed"]):
    # Keep only the defined values
    v = np.asarray(values, dtype=float)
    v = v[~np.isnan(v)]
    # Same seed every time -> same interval every time
    rng = np.random.default_rng(seed)
    # n_boot resamples of the same size, drawn with replacement; the mean of each
    means = rng.choice(v, size=(n_boot, len(v)), replace=True).mean(axis=1)
    # The average and the 2.5th / 97.5th percentiles
    return v.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

# List that collects one row per number we report (becomes summary.csv)
summary_rows = []
# Function: add one row to that list
def add_summary(group, metric, value, lo=np.nan, hi=np.nan, n=np.nan, note=""):
    summary_rows.append({"group": group, "metric": metric, "value": value, "ci_lo": lo, "ci_hi": hi, "n": n, "note": note})

# One table of per-page numbers, built from the stored counts
page_tables = []
for (variant, kind), s in STORE.items():
    # Counts at the official threshold 0.5
    tp = s["tp"][:, I50].astype(float)
    pred = tp + s["fp"][:, I50]
    gt_n = s["gt_pixels"].astype(float)
    # Per-page precision, recall, F1 and IoU (same formulas and eps as main.py)
    p = tp / (pred + eps)
    r = tp / (gt_n + eps)
    f1 = 2 * p * r / (p + r + eps)
    iou = tp / (pred + gt_n - tp + eps)
    # Detection: did the prediction touch the key at all?
    detected = (tp > 0).astype(float)
    # Pages with an empty key have no defined score: mark them NaN
    undefined = gt_n == 0
    for arr in (p, r, f1, iou, detected):
        arr[undefined] = np.nan
    # One DataFrame for this variant and key
    page_tables.append(pd.DataFrame({
        "variant": variant, "gt": kind, "image": s["images"],
        "gt_pixels": s["gt_pixels"], "pred_pixels": pred.astype(int), "tp": tp.astype(int),
        "precision": p, "recall": r, "f1": f1, "iou": iou, "detected": detected,
    }))
per_image = pd.concat(page_tables, ignore_index=True)
# Add the image-level scores of each page (max probability and mean of the top 0.1% of pixels) from scores.csv
per_image = per_image.merge(scores[["variant", "image", "max_prob", "top01_mean"]], on=["variant", "image"], how="left")

# Average + 95% CI for each variant, key and metric
rows_out = []
for (variant, kind), g in per_image.groupby(["variant", "gt"], sort=False):
    group = f"{variant}|{kind}"
    # Number of pages with a defined score
    n_ok = int(g["f1"].notna().sum())
    for col, name in [("f1", "F1 (official-style)"), ("precision", "Precision"), ("recall", "Recall"),
                      ("iou", "IoU"), ("detected", "Detection rate")]:
        m, lo, hi = boot_mean_ci(g[col])
        add_summary(group, name, m, lo, hi, n_ok)
        rows_out.append({"group": group, "metric": name, "n": n_ok, "value": f"{m:.3f}  [{lo:.3f}, {hi:.3f}]"})
# Show it as a table: one row per group, one column per metric
table = pd.DataFrame(rows_out).pivot(index="group", columns="metric", values="value")
table.insert(0, "pages", pd.DataFrame(rows_out).groupby("group")["n"].first())
print("Mean over pages [95% bootstrap CI], threshold 0.5, tampered pages only\n")
print(table[["pages", "F1 (official-style)", "Precision", "Recall", "IoU", "Detection rate"]].to_string())

## 5. Pixel AUC, ROC curve and best-threshold F1 (P6)

**Pixel AUC.**
- **WHAT:** pick a random tampered pixel and a random clean pixel. AUC is the chance that the tampered one has the **higher probability**.
- **WHY:** F1 depends on the 0.5 cut. AUC ignores the cut and asks if the map *ranks* pixels sensibly. AUC = 0.5 means a coin flip; 1.0 means perfect ranking.
- **HOW:** rank all pixels by probability; $AUC = \frac{\text{(sum of ranks of the tampered pixels)} - n_{pos}(n_{pos}+1)/2}{n_{pos}\,n_{neg}}$.
- *Tiny example:* tampered pixels have probabilities 0.9 and 0.4, clean pixels 0.6 and 0.1. Of the 4 (tampered, clean) pairs, three have tampered > clean (0.9>0.6, 0.9>0.1, 0.4>0.1) and one does not (0.4<0.6) → AUC = 3/4 = 0.75.
- We pool all tampered pixels of all pages, and a random 1% of the clean pixels (an unbiased shortcut: far fewer numbers, same expected AUC). The CI resamples **pages**.

**ROC curve:** the same idea drawn as a picture: for every cut, the share of tampered pixels caught (y) against the share of clean pixels wrongly flagged (x). The diagonal is a coin flip.

**Best-threshold F1 (optimistic upper bound).** We try cuts 0.05, 0.10, ... 0.95, compute the official-style F1 for each, and keep the best. **The answer key is used to pick the cut, so this number is too good to be true for a real user.** It only tells us how much F1 we lose just from the 0.5 cut. The CI repeats the whole "pick the best cut" step inside every resample.

⏱ 2–4 min (the bootstrap redoes the AUC 1000 times for 4 groups). ✅ A table with Pixel AUC and best-threshold F1 per group, and a two-panel figure (ROC; F1 against threshold) saved as `figures/roc_curve.png`.

In [ ]:
# Function: AUC from tampered-pixel and clean-pixel probabilities, using ranks (fast)
def auc_rank(pos, neg):
    # Rank all probabilities together (ties get the average rank)
    ranks = rankdata(np.concatenate([pos, neg]))
    n_pos, n_neg = len(pos), len(neg)
    # Mann-Whitney formula = chance that a tampered pixel outranks a clean one
    return (ranks[:n_pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

# Function: bootstrap 95% CI for the pooled pixel AUC, resampling pages
def boot_auc(pos_list, neg_list, n_boot=CONFIG["N_BOOT"], seed=CONFIG["seed"]):
    rng = np.random.default_rng(seed)
    n = len(pos_list)
    aucs = []
    for _ in range(n_boot):
        # Draw n pages with replacement
        idx = rng.integers(0, n, n)
        pos = np.concatenate([pos_list[i] for i in idx])
        neg = np.concatenate([neg_list[i] for i in idx])
        # Skip the rare resample with no tampered pixels at all
        if len(pos) > 0:
            aucs.append(auc_rank(pos, neg))
    return np.percentile(aucs, 2.5), np.percentile(aucs, 97.5)

# Results kept for the figure: pooled pixels per group, and F1 at every threshold
ROC_DATA, F1_CURVES = {}, {}
t0 = time.time()
for (variant, kind), s in STORE.items():
    group = f"{variant}|{kind}"
    # Pooled tampered pixels and sampled clean pixels of all pages
    pos_all = np.concatenate(s["pos"])
    neg_all = np.concatenate(s["neg"])
    # Pixel AUC and its confidence interval
    auc = auc_rank(pos_all, neg_all)
    lo, hi = boot_auc(s["pos"], s["neg"])
    add_summary(group, "Pixel AUC", auc, lo, hi, len(s["pos"]), note=f"{len(pos_all)} tampered px, {len(neg_all)} sampled clean px")
    ROC_DATA[group] = (pos_all, neg_all, auc)

    # Official-style F1 of every page at every threshold: matrix (pages, thresholds)
    tp = s["tp"].astype(float)
    pred = tp + s["fp"]
    gt_n = s["gt_pixels"].astype(float)[:, None]
    p = tp / (pred + eps)
    r = tp / (gt_n + eps)
    f1_mat = 2 * p * r / (p + r + eps)
    # Pages with an empty key are not defined: drop them
    f1_mat = f1_mat[s["gt_pixels"] > 0]
    # Average over pages for each threshold, and pick the best one
    mean_curve = f1_mat.mean(axis=0)
    best = int(np.argmax(mean_curve))
    F1_CURVES[group] = mean_curve
    # Bootstrap: resample pages, and repeat the "pick the best threshold" step each time
    rng = np.random.default_rng(CONFIG["seed"])
    n_pages = f1_mat.shape[0]
    best_boot = [f1_mat[rng.integers(0, n_pages, n_pages)].mean(axis=0).max() for _ in range(CONFIG["N_BOOT"])]
    add_summary(group, "Best-threshold F1 (optimistic)", mean_curve[best], np.percentile(best_boot, 2.5),
                np.percentile(best_boot, 97.5), n_pages, note=f"best threshold {THRESHOLDS[best]}")
    add_summary(group, "Best threshold", THRESHOLDS[best], n=n_pages)
print(f"Done in {time.time() - t0:.0f} s.\n")

# Show the numbers
sm = pd.DataFrame(summary_rows)
view = sm[sm["metric"].isin(["Pixel AUC", "Best-threshold F1 (optimistic)", "Best threshold"])].copy()
view["value [95% CI]"] = [f"{v:.3f}" if np.isnan(lo) else f"{v:.3f}  [{lo:.3f}, {hi:.3f}]"
                          for v, lo, hi in zip(view["value"], view["ci_lo"], view["ci_hi"])]
print(view.pivot(index="group", columns="metric", values="value [95% CI]").to_string())
print("\nF1 at the official threshold 0.5 for comparison:")
print(sm[sm["metric"] == "F1 (official-style)"].set_index("group")["value"].round(3).to_string())

# Figure: left = ROC curves, right = F1 against threshold
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for group, (pos_all, neg_all, auc) in ROC_DATA.items():
    # Points of the ROC curve from the pooled pixels
    labels = np.concatenate([np.ones(len(pos_all)), np.zeros(len(neg_all))])
    fpr, tpr, _ = roc_curve(labels, np.concatenate([pos_all, neg_all]))
    ax1.plot(fpr, tpr, label=f"{group} (AUC {auc:.3f})")
    ax2.plot(THRESHOLDS, F1_CURVES[group], marker="o", markersize=3, label=group)
# Coin-flip diagonal and labels
ax1.plot([0, 1], [0, 1], "k--", label="coin flip (AUC 0.5)")
ax1.set_title("ROC curve, tampered pages (pixels pooled)")
ax1.set_xlabel("False-positive rate (share of clean pixels flagged)")
ax1.set_ylabel("True-positive rate (share of tampered pixels found)")
ax1.legend(loc="lower right")
# Mark the official threshold on the F1 plot
ax2.axvline(CONFIG["threshold"], color="grey", linestyle=":", label="official cut 0.5")
ax2.set_title("Official-style F1 against threshold")
ax2.set_xlabel("Probability threshold")
ax2.set_ylabel("Mean per-page F1")
ax2.legend()
plt.tight_layout()
plt.savefig(f"{EVAL_DIR}/figures/roc_curve.png", dpi=130)
plt.show()

## 6. Image-level scores: can it tell a forged page from an untouched one? (P6)

**WHAT:** give each whole page **one score**, then ask if forged pages score higher than their untouched originals. Two scores: the **maximum probability** anywhere on the page, and the **mean of the top 0.1% of pixels** (less sensitive to a single noisy pixel).

**Image AUC** — the same idea as pixel AUC, but with pages: the chance that a random forged page scores higher than a random original. We pair each forged page with its own original and resample pages together in the bootstrap.
*Tiny example:* forged pages score 0.9, 0.6; originals 0.7, 0.3. Pairs: 0.9>0.7, 0.9>0.3, 0.6>0.3 are wins, 0.6<0.7 is a loss → AUC = 0.75.

**False-alarm rate on originals at 0.5** — of the untouched pages, the share whose score is ≥ 0.5, i.e. the model would raise an alarm although nothing was edited. Originals are the only place we can measure this. **Forged pages flagged at 0.5** is the matching hit rate. A useful detector needs a high hit rate *and* a low false-alarm rate; if both are about equal, the score is not telling pages apart.

⏱ < 30 s. ✅ A table for `png` and `jpg90` (each against the originals). **If the AUC is close to 0.5 and the hit rate is close to the false-alarm rate, the page-level score cannot separate forged from clean pages.**

In [ ]:
# Wide table of image scores: one row per page, one column per variant (score = max probability or top-0.1% mean)
def score_wide(col):
    return scores.pivot(index="image", columns="variant", values=col).sort_index()
wide_max, wide_top = score_wide("max_prob"), score_wide("top01_mean")

# Function: paired bootstrap 95% CI for a statistic of (forged scores, original scores), resampling pages
def boot_pairs(forged, orig, stat, n_boot=CONFIG["N_BOOT"], seed=CONFIG["seed"]):
    rng = np.random.default_rng(seed)
    n = len(forged)
    vals = []
    for _ in range(n_boot):
        # Same page indices for the forged and the original score
        idx = rng.integers(0, n, n)
        vals.append(stat(forged[idx], orig[idx]))
    return np.percentile(vals, 2.5), np.percentile(vals, 97.5)

# The three statistics: image AUC, forged pages flagged at 0.5, originals flagged at 0.5
def stat_auc(forged, orig):
    return roc_auc_score(np.r_[np.ones(len(forged)), np.zeros(len(orig))], np.r_[forged, orig])
def stat_flagged(forged, orig):
    return (forged >= CONFIG["threshold"]).mean()
def stat_false_alarm(forged, orig):
    return (orig >= CONFIG["threshold"]).mean()

img_rows = []
for variant in CONFIG["eval_variants"]:
    for score_name, wide in [("max prob", wide_max), ("top-0.1% mean", wide_top)]:
        # Scores of the forged pages and of their originals, in the same page order
        forged, orig = wide[variant].to_numpy(), wide["orig"].to_numpy()
        group = f"{variant} vs orig"
        out = {"variant": variant, "score": score_name}
        for metric, stat in [("Image AUC", stat_auc), ("Forged pages flagged @0.5", stat_flagged),
                             ("False-alarm rate on originals @0.5", stat_false_alarm)]:
            value = stat(forged, orig)
            lo, hi = boot_pairs(forged, orig, stat)
            add_summary(group, f"{metric} ({score_name})", value, lo, hi, len(forged))
            out[metric] = f"{value:.3f}  [{lo:.3f}, {hi:.3f}]"
        img_rows.append(out)
img_table = pd.DataFrame(img_rows)
print("Image level, 100 forged pages vs their 100 originals [95% bootstrap CI]\n")
print(img_table.to_string(index=False))
print("\nMean max probability: forged png %.3f | forged jpg90 %.3f | originals %.3f"
      % (wide_max["png"].mean(), wide_max["jpg90"].mean(), wide_max["orig"].mean()))

## 7. Summary table, bar chart, and saved files (P6)

**WHY:** one place that gathers every number, with its confidence interval, and the files notebook 05's later parts (and your slides) need.

- `per_image_metrics.csv` — one row per page, variant and answer key (precision, recall, F1, IoU, detected, image scores). P7 joins it with `results.csv` for the breakdowns.
- `summary.csv` — every reported number with its 95% CI and page count.
- `figures/summary_table.png` and `figures/pr_f1_bars.png` — for the slides (the ROC figure was saved in section 5).

⏱ < 30 s. ✅ Two PNG figures shown (a summary table and a bar chart with error bars) and the saved file names. Error bars are 95% CIs; **overlapping bars mean "no clear difference with n = 100".**

In [ ]:
# Save the per-page table and the summary table
per_image.to_csv(f"{EVAL_DIR}/per_image_metrics.csv", index=False)
summary = pd.DataFrame(summary_rows)
summary.to_csv(f"{EVAL_DIR}/summary.csv", index=False)

# Function: look up one number as text "value [lo, hi]" (or just the value)
def cell_text(group, metric):
    row = summary[(summary["group"] == group) & (summary["metric"] == metric)]
    if row.empty:
        return ""
    v, lo, hi = row["value"].iloc[0], row["ci_lo"].iloc[0], row["ci_hi"].iloc[0]
    return f"{v:.3f}" if np.isnan(lo) else f"{v:.3f}\n[{lo:.3f}, {hi:.3f}]"

# Table 1: pixel level
pixel_cols = ["F1 (official-style)", "Precision", "Recall", "IoU", "Detection rate", "Pixel AUC", "Best-threshold F1 (optimistic)"]
groups = [f"{v}|{k}" for v in CONFIG["eval_variants"] for k in CONFIG["gt_kinds"]]
pixel_text = [[cell_text(g, m) for m in pixel_cols] for g in groups]
# Table 2: image level (max probability)
image_cols = ["Image AUC (max prob)", "Forged pages flagged @0.5 (max prob)", "False-alarm rate on originals @0.5 (max prob)"]
image_groups = [f"{v} vs orig" for v in CONFIG["eval_variants"]]
image_text = [[cell_text(g, m) for m in image_cols] for g in image_groups]

# Draw both tables in one figure
fig, (ax_a, ax_b) = plt.subplots(2, 1, figsize=(15, 7), gridspec_kw={"height_ratios": [4, 2]})
for ax, text, rows, cols, title in [
        (ax_a, pixel_text, groups, ["F1\n(official)", "Precision", "Recall", "IoU", "Detection\nrate", "Pixel AUC", "Best-thr F1\n(optimistic)"],
         "Pixel level, 100 forged pages, threshold 0.5 (mean [95% CI]); rows = input | answer key"),
        (ax_b, image_text, image_groups, ["Image AUC", "Forged pages\nflagged @0.5", "False alarms on\noriginals @0.5"],
         "Image level, score = max probability (mean [95% CI])")]:
    ax.axis("off")
    ax.set_title(title, fontsize=11)
    tbl = ax.table(cellText=text, rowLabels=rows, colLabels=cols, loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1, 2.2)
plt.tight_layout()
plt.savefig(f"{EVAL_DIR}/figures/summary_table.png", dpi=130)
plt.show()

# Bar chart: precision, recall and F1 for each group, with 95% CI error bars
fig, ax = plt.subplots(figsize=(10, 5))
metrics = ["Precision", "Recall", "F1 (official-style)"]
width = 0.25
for j, metric in enumerate(metrics):
    vals, errs_lo, errs_hi = [], [], []
    for g in groups:
        row = summary[(summary["group"] == g) & (summary["metric"] == metric)].iloc[0]
        vals.append(row["value"])
        errs_lo.append(row["value"] - row["ci_lo"])
        errs_hi.append(row["ci_hi"] - row["value"])
    ax.bar(np.arange(len(groups)) + (j - 1) * width, vals, width, yerr=[errs_lo, errs_hi], capsize=3, label=metric)
ax.set_xticks(np.arange(len(groups)))
ax.set_xticklabels([g.replace("|", " | ") for g in groups])
ax.set_ylim(0, 1)
ax.set_xlabel("Input | answer key")
ax.set_ylabel("Mean over pages (0 to 1)")
ax.set_title("Precision, recall and F1 at threshold 0.5 (error bars: 95% bootstrap CI)")
ax.legend()
plt.tight_layout()
plt.savefig(f"{EVAL_DIR}/figures/pr_f1_bars.png", dpi=130)
plt.show()

print("Saved in", EVAL_DIR)
print(" - per_image_metrics.csv:", per_image.shape, "| summary.csv:", summary.shape)
print(" - figures:", sorted(os.listdir(f"{EVAL_DIR}/figures")))

# P7 — Breakdowns and bias: where does it do better or worse?

**WHY (links to P6):** P6 said ADCD-Net scores very low on my forged pages overall. The mentor's question is **why**. My forgeries are not all alike: some are numbers, some words; some have capitals; boxes differ in size; banks differ; and in Week 4 many edits were *rendered wrongly* (only 3% of numbers read back correctly). If the model does clearly better on one kind, that points to a cause. If nothing differs, we report "no clear difference with n = 100".

**Problem with F1 alone:** nearly every page has F1 close to 0, so F1 has a "floor" and cannot show differences. So each table also has a more sensitive number:
- **Mean probability inside the box** — WHAT: the average of the model's tamper probability over the box pixels. WHY: it shows whether the model "sees something" there even when it stays below the 0.5 cut. HOW: $\bar p_{box} = \frac{1}{|box|}\sum_{x \in box} p(x)$. *Tiny example:* a 4-pixel box with probabilities 0.1, 0.2, 0.5, 0.2 → 0.25. For scale: the average over a whole page is about 0.095.

| Section | ⏱ Time | Output |
|---|---|---|
| 8. Join scores with `results.csv`, make groups | < 10 s | counts per group |
| 9. Breakdown tables + forest plot + automatic "real difference?" check | < 30 s | tables, `figures/breakdowns.png` |
| 10. PNG vs JPEG-q90 (paired) | < 10 s | table |
| 11. Correlations with SSIM, PSNR, NED, ELA, box size | < 20 s | table |
| 12. Failure gallery | < 30 s | `figures/failure_gallery.png` |

Needs the variables from sections 1–7 in the same session (`per_image`, `STORE`, `results`, `scores`).

## 8. Join the scores with the Week 4 columns and build the groups (P7)

**WHY:** `per_image` knows how well each page was detected; `results.csv` knows what kind of edit it was. We join them on the page name (one row each). The **bank** is the first part of the page path in `results.csv` (e.g. `mfbb_epdf/v1/...` → `mfbb`). Continuous columns are cut into groups: word length into 4 bins, box area (as a share of the page) into **quartiles** (4 groups with 25 pages each). We use the primary setting: **png input, box answer key**.

⏱ < 10 s. ✅ A count per group for each factor (word, number, banks, ...), the box-area quartile ranges, and a line saying the join is one-to-one with 100 rows. Check the **bank counts**: banks with fewer than 10 pages will be flagged "too few to judge".

In [ ]:
# itertools gives all pairs of groups to compare; warnings is used to silence harmless messages
import itertools, warnings
warnings.filterwarnings("ignore")

# One row per forged page: png input scored against the box key (the primary setting)
base = per_image[(per_image["variant"] == "png") & (per_image["gt"] == "box")].copy()
# The stored pixels and counts for that setting
s_png = STORE[("png", "box")]
# Mean probability inside the box, per page (average over the box pixels)
mean_box = pd.Series([p.mean() for p in s_png["pos"]], index=s_png["images"])
# Box area as a share of the page (at model size), per page
box_share = pd.Series(s_png["gt_pixels"] / s_png["page_pixels"], index=s_png["images"])
# Add both to the table
base["mean_prob_box"] = base["image"].map(mean_box)
base["box_share"] = base["image"].map(box_share)

# Bank = first part of the page path, without the "_epdf" ending
results["bank"] = results["page_path"].str.split("/").str[0].str.replace("_epdf", "", regex=False)
# Join the Week 4 columns onto our scores; validate checks that each page matches exactly one row
merged = base.merge(results, left_on="image", right_on="page_id", how="left", validate="one_to_one")
print("Joined:", merged.shape[0], "rows; pages without a match:", int(merged["kind"].isna().sum()))

# Group word length into 4 bins
merged["word_length_group"] = pd.cut(merged["word_length"], bins=[0, 3, 5, 7, 100], labels=["1-3", "4-5", "6-7", "8+"])
# Group box area share into 4 quartiles (25 pages each)
merged["box_area_quartile"] = pd.qcut(merged["box_share"], 4, labels=["Q1 smallest", "Q2", "Q3", "Q4 largest"])
# True/False columns become text, so they behave like any other group label
for col in ["has_upper", "text_correct", "style_kept"]:
    merged[col] = merged[col].astype(str)

# Factor name shown in the tables -> column used for the grouping
FACTORS = {
    "kind": "kind",
    "has_upper (capitals)": "has_upper",
    "word length": "word_length_group",
    "bank": "bank",
    "box area (quartile)": "box_area_quartile",
    "text_correct (edit read back right)": "text_correct",
    "style_kept": "style_kept",
}
# Show the number of pages in each group
for fname, col in FACTORS.items():
    print(f"{fname:38s}", merged[col].value_counts(sort=False).to_dict())
# Show what each box-area quartile means (share of the page, in %)
print("\nBox area quartiles (% of page):")
print((merged.groupby("box_area_quartile", observed=True)["box_share"].agg(["min", "max"]) * 100).round(2).to_string())
# Show the raw word lengths, so the 4 bins can be checked
print("\nRaw word_length counts:", merged["word_length"].value_counts().sort_index().to_dict())

## 9. Breakdown tables, forest plot, and the "real difference?" check (P7)

**WHAT:** for each group we report the number of pages (**n**), and three averages with 95% bootstrap CIs: official-style F1, detection rate, and mean probability inside the box.

**Rule for "real difference":** two groups differ clearly only if their 95% CIs **do not overlap** (this is conservative: non-overlap corresponds to a p-value of roughly 0.005 or smaller). If they overlap, we write "no clear difference with n = 100". Groups with **n < 10 are flagged "too few to judge"** and left out of the comparison. With many comparisons, one or two false positives are expected, so a lone "real difference" should be treated with care and checked for an explanation.

The **forest plot** shows every group as a dot with its CI. Grey dots = too few pages.

⏱ < 30 s. ✅ Seven tables, a list of clear differences (possibly empty!), and `figures/breakdowns.png`.

In [ ]:
# The three numbers reported for every group: column in `merged` -> name in the tables
METRICS = [("f1", "F1 (official-style)"), ("detected", "Detection rate"), ("mean_prob_box", "Mean prob. inside box")]
# Groups with fewer pages than this are "too few to judge"
MIN_N = 10

# Function: one table (one row per group) for one grouping column
def breakdown(col):
    rows = []
    # Go through the groups (observed=True skips empty groups)
    for level, g in merged.groupby(col, observed=True):
        row = {"level": str(level), "n": len(g)}
        # Average and 95% CI of each metric in this group
        for key, name in METRICS:
            m, lo, hi = boot_mean_ci(g[key])
            row[name], row[name + "_lo"], row[name + "_hi"] = m, lo, hi
        rows.append(row)
    return pd.DataFrame(rows).set_index("level")

# Build one table per factor and print it in a readable form
tables = {}
for fname, col in FACTORS.items():
    tbl = breakdown(col)
    tables[fname] = tbl
    show = pd.DataFrame({"n": tbl["n"]})
    for _, name in METRICS:
        show[name] = [f"{m:.3f} [{lo:.3f}, {hi:.3f}]" for m, lo, hi in zip(tbl[name], tbl[name + "_lo"], tbl[name + "_hi"])]
    # Flag the groups that are too small
    show["note"] = ["too few to judge" if n < MIN_N else "" for n in tbl["n"]]
    print(f"\n=== {fname} ===")
    print(show.to_string())

# Automatic check: which pairs of groups have non-overlapping CIs (only groups with n >= MIN_N)?
clear, n_tests = [], 0
for fname, tbl in tables.items():
    ok = tbl[tbl["n"] >= MIN_N]
    for a, b in itertools.combinations(ok.index, 2):
        for _, name in METRICS:
            n_tests += 1
            ra, rb = ok.loc[a], ok.loc[b]
            # CIs do not overlap if one lower end is above the other upper end
            if ra[name + "_lo"] > rb[name + "_hi"] or rb[name + "_lo"] > ra[name + "_hi"]:
                hi_grp, lo_grp = (a, b) if ra[name] > rb[name] else (b, a)
                clear.append(f"{fname}: '{hi_grp}' > '{lo_grp}' on {name}")
print(f"\nCLEAR DIFFERENCES (non-overlapping 95% CIs, {n_tests} comparisons made):")
print("\n".join(" - " + c for c in clear) if clear else " none: no clear difference with n = 100 in any breakdown")

# Save all breakdown tables in one CSV
all_tbl = pd.concat([t.assign(factor=f) for f, t in tables.items()]).reset_index()
all_tbl.to_csv(f"{EVAL_DIR}/breakdowns.csv", index=False)

# Forest plot: one row per group, two panels (F1 and mean probability inside the box)
labels = [f"{r.factor}: {r.level} (n={r.n})" for r in all_tbl.itertuples()]
ypos = np.arange(len(all_tbl))[::-1]
fig, axes = plt.subplots(1, 2, figsize=(14, 0.32 * len(all_tbl) + 2), sharey=True)
for ax, name in zip(axes, ["F1 (official-style)", "Mean prob. inside box"]):
    vals, lo, hi = all_tbl[name].to_numpy(), all_tbl[name + "_lo"].to_numpy(), all_tbl[name + "_hi"].to_numpy()
    small = (all_tbl["n"] < MIN_N).to_numpy()
    # Dots with CI bars; grey for groups that are too small
    ax.errorbar(vals[~small], ypos[~small], xerr=[np.maximum(0, vals - lo)[~small], np.maximum(0, hi - vals)[~small]], fmt="o", color="tab:blue", capsize=3)
    ax.errorbar(vals[small], ypos[small], xerr=[np.maximum(0, vals - lo)[small], np.maximum(0, hi - vals)[small]], fmt="o", color="grey", capsize=3)
    ax.set_title(name + " (mean, 95% CI)")
    ax.set_xlabel(name + " (0 to 1)")
    ax.grid(axis="x", alpha=0.3)
axes[0].set_yticks(ypos)
axes[0].set_yticklabels(labels, fontsize=8)
axes[0].set_ylabel("Group (grey = fewer than 10 pages)")
plt.tight_layout()
plt.savefig(f"{EVAL_DIR}/figures/breakdowns.png", dpi=130)
plt.show()

## 10. PNG vs JPEG-q90 input, paired (H2) (P7)

**WHY:** hypothesis H2 says the pipeline re-saves every page at JPEG quality 100 and throws away the file's own JPEG history, so PNG and JPEG-q90 inputs should score the same. Both versions come from the **same 100 pages**, so we compare them **page by page**: for each page, (JPEG score − PNG score), then average the differences with a bootstrap CI. If the CI contains 0, there is no clear difference. A paired test is sharper than comparing two separate intervals.

**Detection overlap** also shows how many pages are found by both inputs, by only one, or by neither.

**Careful with the image-level AUC in section 6:** the `jpg90` forged pages were compared with `orig` pages that are **PNG**, so that row mixes the effect of the edit with the effect of JPEG compression. The fair image-level comparison is `png` vs `orig` (both uncompressed).

⏱ < 10 s. ✅ A table of mean differences with CIs and a 2x2 detection table.

In [ ]:
# Per-page values for the box key, both inputs, side by side
box_rows = per_image[per_image["gt"] == "box"]
wide = box_rows.pivot(index="image", columns="variant", values=["f1", "detected"])
# Mean probability inside the box for the JPEG input (the PNG one is in `merged`)
s_jpg = STORE[("jpg90", "box")]
mean_box_jpg = pd.Series([p.mean() for p in s_jpg["pos"]], index=s_jpg["images"])

# Page-by-page differences: JPEG minus PNG
diffs = {
    "F1 (official-style)": wide[("f1", "jpg90")] - wide[("f1", "png")],
    "Detection rate": wide[("detected", "jpg90")] - wide[("detected", "png")],
    "Mean prob. inside box": mean_box_jpg - mean_box,
}
rows = []
for name, d in diffs.items():
    # Average difference and its 95% CI
    m, lo, hi = boot_mean_ci(d)
    verdict = "no clear difference (CI contains 0)" if lo <= 0 <= hi else "CLEAR difference (CI excludes 0)"
    rows.append({"metric": name, "mean (jpg90 - png)": f"{m:+.4f}  [{lo:+.4f}, {hi:+.4f}]", "verdict": verdict})
print("Paired differences over 100 pages, box key, threshold 0.5\n")
print(pd.DataFrame(rows).to_string(index=False))

# 2x2 table of detection: found by png? found by jpg90?
det_png, det_jpg = wide[("detected", "png")] > 0, wide[("detected", "jpg90")] > 0
print("\nPages detected (prediction overlaps the box):")
print(f"  by both: {int((det_png & det_jpg).sum())} | only png: {int((det_png & ~det_jpg).sum())} | "
      f"only jpg90: {int((~det_png & det_jpg).sum())} | neither: {int((~det_png & ~det_jpg).sum())}")

## 11. Does the score follow how good the forgery is? Correlations (P7)

**WHAT:** a correlation says whether two numbers move together (+1 together, −1 opposite, 0 unrelated). We use **Spearman's rank correlation** $\rho$ (it compares *orderings*, so it is not thrown off by outliers or by the many F1 values at 0).

**WHY:** SSIM, PSNR and NED come from Week 4 and describe how close the edited word is to the target (NED = text error, lower is better), and ELA ratio is a compression-trace statistic. If the model's score rises with, say, PSNR, the model reacts to how the edit *looks*; if it is unrelated, it does not. We add the box size share too.

**HOW:** rank both columns, then compute the usual correlation of the ranks. *Tiny example:* if the page with the best SSIM also has the highest probability, the second best the second highest, and so on, $\rho = +1$.

Each $\rho$ has a 95% bootstrap CI (resampling pages). A CI that does **not** contain 0 is marked with `*`.

⏱ < 20 s. ✅ A table of 5 predictors x 2 targets. Expect values close to 0 if nothing relates.

In [ ]:
# spearmanr computes Spearman's rank correlation
from scipy.stats import spearmanr

# The Week 4 columns to correlate with the model's score
PREDICTORS = ["ssim", "psnr", "ned", "ela_ratio_tampered", "box_share"]
# The two model-score columns (F1 is mostly 0, so the mean probability inside the box is more sensitive)
TARGETS = [("f1", "F1 (official-style)"), ("mean_prob_box", "Mean prob. inside box")]

# Function: Spearman's rho with a 95% bootstrap CI (resampling pages)
def rho_ci(x, y, n_boot=CONFIG["N_BOOT"], seed=CONFIG["seed"]):
    rng = np.random.default_rng(seed)
    rho = spearmanr(x, y).correlation
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(x), len(x))
        boots.append(spearmanr(x[idx], y[idx]).correlation)
    # NaN appears if a resample is constant; skip those
    return rho, np.nanpercentile(boots, 2.5), np.nanpercentile(boots, 97.5)

rows = []
for col in PREDICTORS:
    row = {"predictor": col}
    for key, name in TARGETS:
        rho, lo, hi = rho_ci(merged[col].to_numpy(float), merged[key].to_numpy(float))
        # Star = the CI does not contain 0
        star = "*" if (lo > 0 or hi < 0) else ""
        row[name] = f"{rho:+.2f} [{lo:+.2f}, {hi:+.2f}]{star}"
    rows.append(row)
print("Spearman correlation with the model's score over 100 forged pages (png, box); * = CI excludes 0\n")
print(pd.DataFrame(rows).to_string(index=False))

## 12. Failure gallery: 3 best, 3 misses, 3 false alarms (P7)

**WHY:** numbers tell us how often; pictures tell us *what it looks like*. Each example is shown as a crop of the page (500 x 500 pixels at full resolution, red outline = the true box), the answer-key box mask, and the model's probability map blown up to the same crop (dark = 0, bright = 1).

- **Best:** the 3 forged pages with the highest F1.
- **Misses:** the 3 forged pages with F1 = 0 where the average probability inside the box is lowest (the model sees nothing at the edit).
- **False alarms:** the 3 **untouched originals** with the most pixels flagged at 0.5. The crop is centred on the page's most suspicious pixel, so the box mask is empty.

Cropping around the box here is only for *display*; it never feeds the model.

⏱ < 30 s. ✅ A table of the 9 pages and `figures/failure_gallery.png` (9 rows x 3 panels). Look at **where** the bright areas are: on the edited word, or somewhere else?

In [ ]:
# The 3 best forged pages (highest F1)
best = merged.sort_values("f1", ascending=False).head(3)
# The 3 worst: F1 lowest, ties broken by the lowest mean probability inside the box
worst = merged.sort_values(["f1", "mean_prob_box"], ascending=[True, True]).head(3)
# The 3 originals with the most pixels flagged at 0.5 (ties broken by the highest max probability)
false_alarms = scores[scores["variant"] == "orig"].sort_values(["pred_pixels", "max_prob"], ascending=False).head(3)

# One entry per gallery row: label, page, which folder holds the page, and the text for the title
examples = []
for r in best.itertuples():
    examples.append({"label": "BEST", "image": r.image, "folder": "tampered", "variant": "png", "r": r,
                     "title": f"BEST {r.image}  F1 {r.f1:.2f}  '{r.source_text}'->'{r.target_text}' ({r.kind})"})
for r in worst.itertuples():
    examples.append({"label": "MISS", "image": r.image, "folder": "tampered", "variant": "png", "r": r,
                     "title": f"MISS {r.image}  F1 {r.f1:.2f}  mean p in box {r.mean_prob_box:.2f}  '{r.source_text}'->'{r.target_text}' ({r.kind})"})
for r in false_alarms.itertuples():
    examples.append({"label": "FALSE ALARM", "image": r.image, "folder": "originals", "variant": "orig", "r": r,
                     "title": f"FALSE ALARM (untouched) {r.image}  {int(r.pred_pixels)} px flagged, max p {r.max_prob:.2f}"})

# Print the table of the chosen pages
print(pd.DataFrame([{"type": e["label"], "image": e["image"], "title": e["title"]} for e in examples]).to_string(index=False))

# Half the width of each crop, in full-resolution pixels
HALF = 250
# Figure: 9 rows x 3 panels
fig, axes = plt.subplots(len(examples), 3, figsize=(12, 3.6 * len(examples)), constrained_layout=True)
for row, e in enumerate(examples):
    # Full-resolution page (BGR -> RGB)
    page = cv2.cvtColor(cv2.imread(f"{SRC_RUN}/{e['folder']}/{e['image']}.png"), cv2.COLOR_BGR2RGB)
    H, W = page.shape[:2]
    # The probability map (model size) blown up to the page size
    prob = np.load(f"{PROBS_DIR}/{e['variant']}/{e['image']}.npy").astype(np.float32)
    prob_full = cv2.resize(prob, (W, H), interpolation=cv2.INTER_LINEAR)
    if e["label"] == "FALSE ALARM":
        # No box on an original: centre on the most suspicious pixel, and use an empty mask
        yy, xx = np.unravel_index(prob.argmax(), prob.shape)
        cx, cy = int(xx * W / prob.shape[1]), int(yy * H / prob.shape[0])
        box_full = np.zeros((H, W), dtype=bool)
        box_rect = None
    else:
        # Centre on the box (positions from results.csv, in full-resolution pixels)
        r = e["r"]
        cx, cy = int((r.x1 + r.x2) / 2), int((r.y1 + r.y2) / 2)
        box_full = cv2.imread(f"{SRC_RUN}/masks/box/{e['image']}.png", 0) > 0
        box_rect = (r.x1, r.y1, r.x2, r.y2)
    # Crop window, kept inside the page
    x0, x1 = max(0, cx - HALF), min(W, cx + HALF)
    y0, y1 = max(0, cy - HALF), min(H, cy + HALF)
    # Panel 1: page crop (with the red box outline for forged pages)
    axes[row, 0].imshow(page[y0:y1, x0:x1], extent=(x0, x1, y1, y0))
    if box_rect is not None:
        axes[row, 0].add_patch(plt.Rectangle((box_rect[0], box_rect[1]), box_rect[2] - box_rect[0], box_rect[3] - box_rect[1],
                                             fill=False, edgecolor="red", linewidth=1.5))
    axes[row, 0].set_title(e["title"], fontsize=8, loc="left")
    # Panel 2: answer-key box mask
    axes[row, 1].imshow(box_full[y0:y1, x0:x1], cmap="gray", vmin=0, vmax=1, extent=(x0, x1, y1, y0))
    axes[row, 1].set_title("Box mask (answer key)" if box_rect else "Box mask (empty: untouched page)", fontsize=8)
    # Panel 3: probability map
    im = axes[row, 2].imshow(prob_full[y0:y1, x0:x1], cmap="magma", vmin=0, vmax=1, extent=(x0, x1, y1, y0))
    axes[row, 2].set_title(f"Tamper probability (max in crop {prob_full[y0:y1, x0:x1].max():.2f})", fontsize=8)
    # Axis labels on every panel (pixels of the full-resolution page)
    for col in range(3):
        axes[row, col].set_xlabel("x (full-res pixels)", fontsize=7)
        axes[row, col].set_ylabel("y (full-res pixels)", fontsize=7)
        axes[row, col].tick_params(labelsize=6)
# One colour bar for all probability panels
fig.colorbar(im, ax=axes[:, 2].tolist(), fraction=0.03, pad=0.02, label="tamper probability")
plt.savefig(f"{EVAL_DIR}/figures/failure_gallery.png", dpi=90)
plt.show()

## Which breakdowns show a real difference? (P7 conclusion)

**Overall level first.** Even the best groups are far below the 0.5 cut: the mean probability inside the box is 0.15 to 0.31, against about 0.095 for an average page. So the model reacts a little to the edited area (about 2x a typical page pixel) but never confidently. This fits the pixel AUC of 0.85 with an F1 near 0.

**Breakdowns with non-overlapping 95% CIs (78 comparisons made, so one or two chance hits are possible):**
- **Box area:** the largest quartile (box 0.28% to 2.52% of the page) beats Q1, Q2 and Q3 on F1 (0.074 [0.007, 0.146] vs 0.000) and on detection (0.20 vs 0.00), and beats Q1 on mean probability (0.249 vs 0.166). Box size also correlates with the score (Spearman +0.32 with F1, +0.36 with mean probability, both CIs exclude 0). This is the clearest continuous pattern and is consistent with H1 (resolution): small boxes shrink to a few pixels. It is **not proof**; the tiled test in notebook 06 is the real check.
- **Capitals (`has_upper`):** True beats False on F1, detection and mean probability (0.213 [0.188, 0.242] vs 0.167 [0.154, 0.181]).
- **`kind` (word vs number) and `text_correct`:** words and correctly rendered edits beat numbers and wrongly rendered edits on F1 and detection. On mean probability, word vs number overlaps (0.207 [0.186, 0.235] vs 0.171 [0.156, 0.187]): **no clear difference** on the more sensitive measure.
- **Bank:** only on mean probability, `bimb` and `cibb` above `arbk` and `pbbe`; F1 and detection show nothing. 9 of 14 banks have under 10 pages, so most banks are **too few to judge**.

**No clear difference:** word length (all bins overlap), `style_kept`, and SSIM, PSNR and ELA ratio (correlations near 0; their CIs contain 0). NED has a small negative correlation with F1 (-0.16 [-0.24, -0.08]): edits with more text errors score slightly lower.

**Two cautions that matter more than any single p-value:**
1. **The F1 and detection "differences" partly come from zeros.** Groups with no detections at all (numbers, no capitals, small boxes) get a confidence interval of exactly [0, 0], so any group with even a few hits looks "non-overlapping". With 33 to 40 pages and 0 hits, the honest ceiling is still a detection rate of about 8%.
2. **The factors are tangled.** All 5 pages the model detected (detection rate 0.20 in Q4 = 5/25, 0.083 in has_upper True = 5/60, 0.075 in words = 5/67) are words with capitals, a correctly rendered edit and a large box. There are 33 numbers, 40 pages without capitals and 34 wrongly rendered edits, so these three groups are probably mostly the same pages (inferred from the counts, not cross-tabulated). With 100 pages we **cannot separate** "number vs word", "capitals", "text_correct" and "box size".

**PNG vs JPEG-q90 (H2), paired:** JPEG scored slightly higher: F1 +0.012 [+0.002, +0.028], mean probability inside the box +0.024 [+0.015, +0.033] (both CIs exclude 0), detection +0.04 [0.00, +0.08] (not clear). Both are near zero in practice (F1 0.018 vs 0.031), so JPEG history is not what rescues or breaks the score, but H2's "should score the same" is only approximately true.

**Failure gallery (cell 12 output):** the 3 best pages are capitalised words (`CIMBBANK`->`DIVIDEND`, `ISLAM`->`TOTAL`, `ISLAM`->`ASSET`, F1 0.45 to 0.63); 2 of the 3 misses are numbers. **Pages 0021 and 0075 appear in both the best list and the false-alarm list** (their untouched originals also have about 4,500 pixels flagged), so on those pages the model may flag the area whatever the edit. Check this before calling those "successes".

In [ ]:
# zipfile makes .zip archives (notebook 05 had not imported it yet)
import zipfile

# Folder that holds everything notebook 05 wrote (CSVs, config.json and figures)
ZIP_SRC = EVAL_DIR
# Name of the zip file, using the folder name, saved in /kaggle/working so it shows in the Output tab
ZIP_PATH = f"{WORK}/{EVAL_NAME}.zip"
# The figures we expect inside the zip (three from P6, two from P7)
EXPECTED_FIGS = ["summary_table.png", "roc_curve.png", "pr_f1_bars.png", "breakdowns.png", "failure_gallery.png"]

# Check that every expected figure exists before zipping
for fig_name in EXPECTED_FIGS:
    # Full path of this figure
    fig_path = f"{ZIP_SRC}/figures/{fig_name}"
    # Stop with a clear message if a figure is missing
    if not os.path.isfile(fig_path):
        raise FileNotFoundError(f"Missing figure, not zipped: {fig_path}")

# Count the files we add to the zip
n_added = 0
# Open a new zip file for writing, with compression
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    # Walk through every folder and file inside the output folder
    for folder, _, fnames in os.walk(ZIP_SRC):
        for fname in fnames:
            # Full path of this file on disk
            full_path = os.path.join(folder, fname)
            # Store it inside the zip under the run name, keeping the folder structure
            arc_name = os.path.join(EVAL_NAME, os.path.relpath(full_path, ZIP_SRC))
            zf.write(full_path, arc_name)
            n_added += 1

# Re-open the zip and list the figures inside it, to prove they are there
with zipfile.ZipFile(ZIP_PATH) as zf:
    inside = [n for n in zf.namelist() if "/figures/" in n]
# Print the result: file count, zip size, and each figure found inside
print(f"Zipped {n_added} files -> {ZIP_PATH} ({os.path.getsize(ZIP_PATH) / 1e6:.1f} MB)")
print("Figures inside the zip:")
for name in inside:
    print("  ", name)

In [ ]:
from IPython.display import FileLink
FileLink("adcd_eval_20261026_1406.zip")